In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a GPU program that raises a square matrix $A$ of size $N \times N$ to an integer power $P$.<br/>
    The <code>solve</code> function receives a flattened input matrix <code>input</code> (row-major order), an empty output matrix <code>output</code> of the same size, the dimension <code>N</code>, and the exponent <code>P</code>.<br/>
    You must compute $\text{output} = A^{P}$ where matrix multiplication is standard dense multiplication over 32-bit floating point numbers.
  </p>

  <h2>Implementation Requirements</h2>
  <ul>
    <li>External libraries are <strong>not</strong> permitted.</li>
    <li>The <code>solve</code> function signature must remain unchanged.</li>
    <li>The final result must be written to the <code>output</code> array in row-major order.</li>
  </ul>

  <h2>Example 1:</h2>
  <pre>
  Input:
    input  = [[1.0, 2.0],
              [3.0, 4.0]]
    N      = 2
    P      = 3
  Output:
    output = [[37.0, 54.0],
              [81.0, 118.0]]
  </pre>

  <h2>Example 2:</h2>
  <pre>
  Input:
    input  = [[1.0, 0.0, 2.0],
              [0.0, 1.0, 0.0],
              [3.0, 0.0, 0.0]]
    N      = 3
    P      = 2
  Output:
    output = [[7.0, 0.0, 2.0],
              [0.0, 1.0, 0.0],
              [3.0, 0.0, 6.0]]
  </pre>

  <h2>Constraints</h2>
  <ul>
    <li>$1 \le N \le 1024$</li>
    <li>$1 \le P \le 20$</li>
    <li>Elements of <code>input</code> satisfy $-10.0 \le A_{ij} \le 10.0$</li>

  <li>Performance is measured with <code>N</code> = 512</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers
extern "C" void solve(const float* input, float* output, int N, int P) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, N: cute.Int32, P: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input is a tensor on the GPU
@jax.jit
def solve(input: jax.Array, N: int, P: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    P: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, P: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, P: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Matrix Power", atol=1e-04, rtol=1e-04, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, input: torch.Tensor, output: torch.Tensor, N: int, P: int):
        """
        Matrix power implementation using PyTorch.
        Raises an N x N matrix to integer power P.
        """
        assert input.dtype == torch.float32
        assert output.dtype == torch.float32
        assert input.shape == output.shape == (N * N,)
        assert P >= 1

        mat = input.view(N, N)
        result = torch.linalg.matrix_power(mat, P).float()
        output[:] = result.reshape(-1)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
            "P": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 2
        P = 3
        input_data = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device="cuda", dtype=dtype).flatten()
        output_data = torch.zeros((2, 2), device="cuda", dtype=dtype).flatten()

        return {
            "input": input_data,
            "output": output_data,
            "N": N,
            "P": P,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_cases = []

        # Test case 1: example 2x2 power 3
        test_cases.append(
            {
                "input": torch.tensor(
                    [[1.0, 2.0], [3.0, 4.0]], device="cuda", dtype=dtype
                ).flatten(),
                "output": torch.zeros((2, 2), device="cuda", dtype=dtype).flatten(),
                "N": 2,
                "P": 3,
            }
        )

        # Test case 2: identity 3x3 power 5
        test_cases.append(
            {
                "input": torch.eye(3, device="cuda", dtype=dtype).flatten(),
                "output": torch.zeros((3, 3), device="cuda", dtype=dtype).flatten(),
                "N": 3,
                "P": 5,
            }
        )

        # Test case 3: random 5x5 power 2
        test_cases.append(
            {
                "input": torch.empty((5, 5), device="cuda", dtype=dtype)
                .uniform_(-5.0, 5.0)
                .flatten(),
                "output": torch.zeros((5, 5), device="cuda", dtype=dtype).flatten(),
                "N": 5,
                "P": 2,
            }
        )

        # Test case 4: random 16x16 power 3
        test_cases.append(
            {
                "input": torch.empty((16, 16), device="cuda", dtype=dtype)
                .uniform_(-1.0, 1.0)
                .flatten(),
                "output": torch.zeros((16, 16), device="cuda", dtype=dtype).flatten(),
                "N": 16,
                "P": 3,
            }
        )

        # Test case 5: random 8x8 power 4
        test_cases.append(
            {
                "input": torch.empty((8, 8), device="cuda", dtype=dtype)
                .uniform_(-10.0, 10.0)
                .flatten(),
                "output": torch.zeros((8, 8), device="cuda", dtype=dtype).flatten(),
                "N": 8,
                "P": 4,
            }
        )

        # Test case 6: random 10x10 power 1
        test_cases.append(
            {
                "input": torch.empty((10, 10), device="cuda", dtype=dtype)
                .uniform_(-2.0, 2.0)
                .flatten(),
                "output": torch.zeros((10, 10), device="cuda", dtype=dtype).flatten(),
                "N": 10,
                "P": 1,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 512
        P = 3
        return {
            "input": torch.empty((N, N), device="cuda", dtype=dtype)
            .uniform_(-10.0, 10.0)
            .flatten(),
            "output": torch.zeros((N, N), device="cuda", dtype=dtype).flatten(),
            "N": N,
            "P": P,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
